In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config.config import MODELS_BASELINE

from src.features.build_features import build_features_dataframe, feature_columns 
from src.repository.parquet_repository import ParquetRepository

from src.models.LightGBMRegressorAnomaly import LightGBMRegressorAnomaly
from src.models.XGBoostRegressorAnomaly import XGBoostRegressorAnomaly
from src.features.features import (sort_by_forecast_date, build_date_based_cv_splits)
from sklearn.model_selection import TimeSeriesSplit


In [2]:
# Exemplo futuro de leitura para o LightGBM, com h=15
repo = ParquetRepository(base_path=ROOT / "data")
df = repo.load("features/anomaly_predictions.parquet")
#df_15 = df[df["horizon_days"] == 7]
#df_15 = df[df["horizon_days"] == 15]
df_15 = df[(df["horizon_days"] == 7) | (df["horizon_days"] == 15)] #.isin(['A', 'B', 'C'])
X_train = df_15.loc[df_15["split"] == "train", feature_columns(df_15)]
y_train = df_15.loc[df_15["split"] == "train", "quantity"]
print(X_train.shape, y_train.shape)


[OK] Arquivo carregado: d:\PROJETOS\git_repo\PharmaDemand-ml\data\features\anomaly_predictions.parquet
(3712, 40) (3712,)


In [3]:
colunas = {"anomaly_score_mean7", "anomaly_score_mean15"}

if colunas.issubset(df.columns):
    print("As duas colunas existem!")
else:
    faltantes = colunas - set(df.columns)
    print(f"Colunas ausentes: {faltantes}")

Colunas ausentes: {'anomaly_score_mean7', 'anomaly_score_mean15'}


In [4]:
print (df.dtypes)

date                    datetime64[ns]
category                        object
horizon_days                     int64
forecast_origin         datetime64[ns]
split                           object
quantity                       float64
year                             int32
month                            int32
quarter                          int32
weekday                          int32
dayofyear                        int32
weekofyear                       int64
is_weekend                       int64
weekday_sin                    float64
weekday_cos                    float64
annual_sin                     float64
annual_cos                     float64
qty_lag_origin                 float64
qty_lag_origin_1               float64
qty_lag_origin_7               float64
qty_lag_origin_14              float64
qty_lag_origin_28              float64
qty_lag_origin_56              float64
qty_roll_mean_7                float64
qty_roll_std_7                 float64
qty_roll_sum_7           

In [5]:
X_test = df_15.loc[df_15["split"] == "test", feature_columns(df_15)]
y_test = df_15.loc[df_15["split"] == "test", "quantity"]
print(X_test.shape, y_test.shape)

(366, 40) (366,)


In [6]:
X_test.head()

,year,month,quarter,weekday,dayofyear,weekofyear,is_weekend,weekday_sin,weekday_cos,annual_sin,...,qty_roll_std_56,qty_mean_ratio_7_28,qty_mean_delta_7_28,qty_last_ratio_28,qty_last_zscore_28,qty_cv_28,qty_zero_ratio_28,qty_mean_ratio_14_56,anomaly_score,is_anomaly
1860,2019,4,2,1,99,15,0,0.781831,0.623490,0.991269,...,11.555353,0.792710,-6.296429,0.790123,-0.575679,0.364572,0.0,0.856451,0.107515,0
1861,2019,4,2,2,100,15,0,0.974928,-0.222521,0.988854,...,11.684584,0.769873,-6.885714,0.611602,-1.028732,0.377550,0.0,0.837558,0.116822,0
1862,2019,4,2,3,101,15,0,0.433884,-0.900969,0.986146,...,11.632650,0.859483,-4.350000,1.259806,0.748972,0.346883,0.0,0.885301,0.107962,0
1863,2019,4,2,4,102,15,0,-0.433884,-0.900969,0.983147,...,11.377566,0.867529,-4.060714,0.822090,-0.506049,0.351566,0.0,0.861570,0.026718,0
1864,2019,4,2,5,103,15,1,-0.974928,-0.222521,0.979857,...,11.264431,0.877889,-3.603571,0.447295,-1.500537,0.368338,0.0,0.875710,0.143076,0


In [7]:
print (df_15.dtypes)

date                    datetime64[ns]
category                        object
horizon_days                     int64
forecast_origin         datetime64[ns]
split                           object
quantity                       float64
year                             int32
month                            int32
quarter                          int32
weekday                          int32
dayofyear                        int32
weekofyear                       int64
is_weekend                       int64
weekday_sin                    float64
weekday_cos                    float64
annual_sin                     float64
annual_cos                     float64
qty_lag_origin                 float64
qty_lag_origin_1               float64
qty_lag_origin_7               float64
qty_lag_origin_14              float64
qty_lag_origin_28              float64
qty_lag_origin_56              float64
qty_roll_mean_7                float64
qty_roll_std_7                 float64
qty_roll_sum_7           

In [8]:
df_15["date"] = pd.to_datetime(df_15["date"])

df_train_lgbm = df_15.loc[df_15["split"] == "train"].copy()
df_test_lgbm = df_15.loc[df_15["split"] == "test"].copy()

print("Treinamento:", df_train_lgbm.shape)
print("Teste:", df_test_lgbm.shape)  

Treinamento: (3712, 46)
Teste: (366, 46)


In [9]:
print(df_15["split"].value_counts(dropna=False))

print("Treinamento:")
print(df_train_lgbm["date"].min(), df_train_lgbm["date"].max())

print("Teste:")
print(df_test_lgbm["date"].min(), df_test_lgbm["date"].max())

print("Categorias:")
print(df_15["category"].unique())

split
train    3712
test      366
Name: count, dtype: int64
Treinamento:
2014-03-06 00:00:00 2019-04-08 00:00:00
Teste:
2019-04-09 00:00:00 2019-10-08 00:00:00
Categorias:
['N02BE']


In [10]:
target_col = "quantity"

LEAKAGE_SAFE_NUMERIC_FEATURES = [
    "qty_lag_origin", 
    "qty_lag_origin_7", 
    "qty_roll_mean_7",
    "qty_roll_std_7", 
    "qty_roll_mean_28", 
    "qty_roll_std_28",
    "qty_mean_ratio_7_28", 
    "qty_last_zscore_28", 
    "qty_zero_ratio_28",
    "weekday", 
    "annual_sin", 
    "annual_cos",
]

#LEAKAGE_SAFE_CATEGORICAL_FEATURES = ["Product", "Region"] nao tem

LAGGED_ANOMALY_FEATURES = [
    "anomaly_score",    
    "anomaly_score_mean7",
    "anomaly_score_mean15",
    "is_anomaly_rate7",
    "is_anomaly_rate7",
    "is_anomaly"
]

LEAKAGE_SAFE_HYBRID_NUMERIC_FEATURES = (
    LEAKAGE_SAFE_NUMERIC_FEATURES + LAGGED_ANOMALY_FEATURES
)

In [11]:

numeric_features = [
    "qty_lag_origin", 
    "qty_lag_origin_7", 
    "qty_roll_mean_7",
    "qty_roll_std_7", 
    "qty_roll_mean_28", 
    "qty_roll_std_28",
    "qty_mean_ratio_7_28", 
    "qty_last_zscore_28", 
    "qty_zero_ratio_28",
    "weekday", 
    "annual_sin", 
    "annual_cos",
]
#categorical_features = LEAKAGE_SAFE_CATEGORICAL_FEATURES.copy()



In [12]:

anomaly_model = LightGBMRegressorAnomaly(
    target_col=target_col,
    numeric_features=numeric_features,
    categorical_features=None,
    model_dir=MODELS_BASELINE,
    random_state=3,
    n_splits=5,
    scoring="neg_root_mean_squared_error"
)

In [13]:
grid_result = anomaly_model.gridSearchFind(df_train_lgbm)

Fitting 5 folds for each of 96 candidates, totalling 480 fits


In [14]:
print(f"Best Params: {anomaly_model.best_params}")

model_path = anomaly_model.save_model(filename="lightgbm_anomalies_model.joblib")
params_path = anomaly_model.save_best_params(filename="lightgbm_anomalies_metrics.json")

print("Model saved at:", model_path)
print("Best params saved at:", params_path)

Best Params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__max_depth': 5, 'model__min_child_samples': 20, 'model__n_estimators': 500, 'model__num_leaves': 15, 'model__reg_lambda': 10.0, 'model__subsample': 0.8, 'model__subsample_freq': 1}
Model saved at: D:\PROJETOS\git_repo\PharmaDemand-ml\models\baseline\lightgbm_anomalies_model.joblib
Best params saved at: D:\PROJETOS\git_repo\PharmaDemand-ml\models\baseline\lightgbm_anomalies_metrics.json


In [15]:
df_train_lgbm = df_train_lgbm[(df_train_lgbm["horizon_days"])==15]

metrics = anomaly_model.evaluate(df_train_lgbm)

print(metrics)

{'mae': 7.4000450937618405, 'rmse': np.float64(9.579677250591802), 'r2': 0.6348669354659179}


In [16]:
df_test_lgbm = df_test_lgbm[(df_test_lgbm["horizon_days"])==15]

metrics2 = anomaly_model.evaluate(df_test_lgbm)
print(metrics2)

{'mae': 7.218112341727426, 'rmse': np.float64(9.09046842163069), 'r2': 0.31862558754056214}


In [17]:
model = XGBoostRegressorAnomaly(
    target_col=target_col,
    numeric_features=numeric_features,
    categorical_features=None,
    model_dir=MODELS_BASELINE,
    random_state=3,
    n_splits=3,
    scoring="neg_root_mean_squared_error"
)

In [18]:
grid_result = model.gridSearchFind(df_train_lgbm)

print(f"Best Params: {anomaly_model.best_params}")

Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best Params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__max_depth': 5, 'model__min_child_samples': 20, 'model__n_estimators': 500, 'model__num_leaves': 15, 'model__reg_lambda': 10.0, 'model__subsample': 0.8, 'model__subsample_freq': 1}


In [19]:
print(f"Best Params: {model.best_params}")

model_path = model.save_model(filename="XGBoost_model.joblib")
params_path = model.save_best_params(filename="lXGBoost_metrics.json")

print("Model saved at:", model_path)
print("Best params saved at:", params_path)

Best Params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__max_depth': 3, 'model__min_child_weight': 5, 'model__n_estimators': 300, 'model__reg_lambda': 5.0, 'model__subsample': 0.8}
Model saved at: D:\PROJETOS\git_repo\PharmaDemand-ml\models\baseline\XGBoost_model.joblib
Best params saved at: D:\PROJETOS\git_repo\PharmaDemand-ml\models\baseline\lXGBoost_metrics.json


In [20]:
metricsXG = model.evaluate(df_train_lgbm)

print(metricsXG)

{'mae': 8.01602915814904, 'rmse': 10.64149855348879, 'r2': 0.549437557071667}


In [21]:
metricsXG2 = model.evaluate(df_test_lgbm)
print(metricsXG2)

{'mae': 7.0514512723495395, 'rmse': 8.993376066871836, 'r2': 0.3331029381864715}
